In [ ]:
import os, subprocess, sys, time
os.chdir('/kaggle/working')
print('ETAPE1: clone + deps')
r = subprocess.run(['git','clone','--depth','1','https://github.com/comfyanonymous/ComfyUI.git'],capture_output=True,text=True)
print('clone rc=', r.returncode, r.stderr[-300:] if r.returncode else '')
r = subprocess.run([sys.executable,'-m','pip','install','-q','-r','ComfyUI/requirements.txt'],capture_output=True,text=True)
print('req rc=', r.returncode, r.stderr[-300:] if r.returncode else '')
r = subprocess.run([sys.executable,'-m','pip','install','-q','aiohttp'],capture_output=True,text=True)
print('aiohttp rc=', r.returncode)
print('ETAPE1 FINIE')

In [ ]:
print('ETAPE2: telechargement assets depuis le manifest')
import os, subprocess
# __MANIFEST_JSON__ -- injecte par patch_tunnel.py selon le profil choisi
MANIFEST = __MANIFEST_JSON__
def _dl(d, files):
    os.makedirs(d, exist_ok=True)
    for name, url in files.items():
        dst = os.path.join(d, name)
        if os.path.exists(dst) and os.path.getsize(dst) > 1e6:
            continue
        src = url
        if url.startswith('/kaggle/input/'):
            import shutil; shutil.copy(url, dst); print('dataset:', name)
        else:
            r = subprocess.run(['wget','-q','--show-progress','-O',dst,url],capture_output=True,text=True)
            print(name, 'rc=', r.returncode, (r.stderr or '')[-150:])
base='/kaggle/working/ComfyUI/models'
_dl(base+'/checkpoints', MANIFEST.get('checkpoints', {}))
_dl(base+'/loras', MANIFEST.get('loras', {}))
for grp in (MANIFEST.get('models_extra') or {}).values():
    if 'files' in grp:
        _dl(base+'/'+grp['dir'], grp['files'])
for repo in MANIFEST.get('custom_nodes', []):
    dest='/kaggle/working/ComfyUI/custom_nodes/'+repo.rstrip('/').split('/')[-1]
    if not os.path.exists(dest):
        subprocess.run(['git','clone','--depth','1',repo,dest],capture_output=True)
        req=os.path.join(dest,'requirements.txt')
        if os.path.exists(req):
            subprocess.run(['pip','install','-q','-r',req],capture_output=True)
        print('node:', repo.rsplit('/',1)[-1])
print('ckpts:', os.listdir(base+'/checkpoints'))
print('loras:', os.listdir(base+'/loras') if os.path.exists(base+'/loras') else [])
print('ETAPE2 FINIE')


In [ ]:
print('ETAPE3: comfy + proxy token-gate')
import os, threading, subprocess, sys, time, asyncio, signal
from aiohttp import web, ClientSession
TOKEN = os.environ.get('HEPHAISTOS_TOKEN') or '__KAGGLE_COMFY_TOKEN__'
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret('HEPHAISTOS_TOKEN') or TOKEN
    print('token via kaggle secrets')
except Exception as e:
    print('token via env/placeholder:', type(e).__name__)
comfy = subprocess.Popen([sys.executable,'main.py','--listen','127.0.0.1','--port','8188','--disable-cuda-malloc'],
    cwd='/kaggle/working/ComfyUI')
async def proxy(req):
    if req.headers.get('X-Hephaistos-Token') != TOKEN:
        return web.Response(status=401, text='bad token')
    _last[0] = time.time()
    async with ClientSession() as s:
        url = 'http://127.0.0.1:8188' + req.path_qs
        body = await req.read() if req.can_read_body else None
        async with s.request(req.method, url, data=body,
            headers={k:v for k,v in req.headers.items() if k.lower() not in ('host','x-hephaistos-token')}) as up:
            return web.Response(status=up.status, body=await up.read(),
                headers={k:v for k,v in up.headers.items() if k.lower() not in ('content-encoding','content-length','transfer-encoding')})
async def shutdown(req):
    if req.headers.get('X-Hephaistos-Token') != TOKEN:
        return web.Response(status=401, text='bad token')
    threading.Timer(2.0, lambda: os.kill(os.getpid(), signal.SIGTERM)).start()
    return web.Response(text='extinction dans 2s')
async def main():
    app = web.Application()
    app.router.add_route('*','/shutdown',shutdown)
    app.router.add_route('*','/{path:.*}',proxy)
    r = web.AppRunner(app); await r.setup()
    await web.TCPSite(r,'0.0.0.0',8080).start()
    print('proxy pret :8080')
threading.Thread(target=lambda: asyncio.run(main()), daemon=True).start()
MAX_IDLE_S = float(os.environ.get('KERNEL_MAX_IDLE_SECONDS','600'))
_last = [time.time()]
def _idle_guard():
    while True:
        time.sleep(30)
        if time.time() - _last[0] > MAX_IDLE_S:
            print('IDLE', MAX_IDLE_S, 's -> auto-extinction', flush=True); os.kill(os.getpid(), signal.SIGTERM)
threading.Thread(target=_idle_guard, daemon=True).start()
print('auto-extinction si idle >', MAX_IDLE_S, 's')
time.sleep(20)
print('comfy poll:', comfy.poll(), '(None = vivant)')
print('ETAPE3 FINIE')


In [ ]:
print('ETAPE4: tunnel')
import subprocess, re, time, threading, os
import json as _json, os as _os
_os.makedirs('/kaggle/working/.cloudflared', exist_ok=True)
_creds = _json.loads(r'''__CF_CREDS_JSON__''')
open('/kaggle/working/.cloudflared/tunnel.json','w').write(_json.dumps(_creds))
_cfg = chr(10).join(['tunnel: __CF_TUNNEL_ID__',
 'credentials-file: /kaggle/working/.cloudflared/tunnel.json',
 'ingress:',
 '  - hostname: __CF_HOSTNAME__',
 '    service: http://127.0.0.1:8188',
 '  - service: http_status:404', ''])
open('/kaggle/working/cf-config.yml','w').write(_cfg)
print('cf config ecrite: __CF_HOSTNAME__ -> :8188')
if not os.path.exists('/kaggle/working/cloudflared'):
    r = subprocess.run(['wget','-q','https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64','-O','/kaggle/working/cloudflared'],capture_output=True,text=True)
    print('cloudflared dl rc=', r.returncode)
subprocess.run(['chmod','+x','/kaggle/working/cloudflared'],check=False)
_cf_cmd = ['/kaggle/working/cloudflared','tunnel','--no-autoupdate','--config','/kaggle/working/cf-config.yml','run','__CF_TUNNEL_ID__']
proc = subprocess.Popen(_cf_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for i, line in enumerate(proc.stdout):
    print('CF:', line.strip()[:160])
    if 'Registered tunnel connection' in line:
        print('>>> TUNNEL NOMME ACTIF: https://__CF_HOSTNAME__')
        break
    if i > 60: break


In [ ]:
# WATCHDOG + KEEP-ALIVE : relance cloudflared ET comfy s'ils meurent
import time, subprocess, os, signal, re, threading, sys
cf = [None]
def start_cf():
    _cf_cmd = ['/kaggle/working/cloudflared','tunnel','--no-autoupdate','--config','/kaggle/working/cf-config.yml','run','__CF_TUNNEL_ID__']
    cf[0] = subprocess.Popen(_cf_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    def pump():
        for line in cf[0].stdout:
            if 'Registered tunnel connection' in line:
                print('>>> TUNNEL ACTIF: https://__CF_HOSTNAME__', flush=True)
            elif ' ERR ' in line or 'error' in line.lower():
                print('CF-ERR:', line.strip()[:140], flush=True)
    threading.Thread(target=pump, daemon=True).start()
def start_comfy():
    global comfy
    comfy = subprocess.Popen([sys.executable,'main.py','--listen','127.0.0.1','--port','8188','--disable-cuda-malloc'],
        cwd='/kaggle/working/ComfyUI')
print('watchdog actif', flush=True)
while True:
    if cf[0] is None or cf[0].poll() is not None:
        print('WATCHDOG: cloudflared mort -> redemarrage', flush=True)
        start_cf()
    if comfy.poll() is not None:
        print('WATCHDOG: comfy mort -> redemarrage', flush=True)
        start_comfy()
    time.sleep(20)
